# Building an Image Recognition System for Quality Control

**Task 13 — Binary Image Classification for Manufacturing Defect Detection**

## 1. Case Overview

Manufacturing companies inspect products before shipping them to customers, deciding whether each item is **acceptable** or **defective**. Manual inspection becomes difficult when:

- Thousands of products are manufactured every day
- Defects are small or hard to notice
- Inspectors get tired after reviewing many products
- Different inspectors make different decisions
- Products move quickly through the production line

In this notebook, we build a **binary image classification system** that automatically examines an image of a manufactured product and predicts whether it is **OK (acceptable)** or **Defective**.

**Dataset used:** [Casting Product Image Data for Quality Inspection](https://www.kaggle.com/datasets/ravirajsinh45/real-life-industrial-dataset-of-casting-product) — grayscale images (300x300, later resized) of submersible pump impellers, labeled `ok_front` and `def_front`.

**What this notebook covers:**
1. Environment setup & dataset download
2. Exploratory Data Analysis (EDA)
3. Data preprocessing & augmentation
4. Baseline CNN model (built from scratch)
5. Transfer learning model (MobileNetV2)
6. Model comparison & evaluation (accuracy, precision, recall, F1, ROC-AUC, confusion matrix)
7. Error analysis (misclassified images)
8. Saving & exporting the final model for deployment
9. Business summary & recommendations


## 2. Environment Setup

Run this cell first. It installs/imports everything needed. Works in Google Colab or a local Jupyter environment with a GPU (recommended) or CPU (slower but fine for this dataset size).


In [ ]:
# If running in Colab, uncomment the line below to make sure TensorFlow is available
# !pip install -q tensorflow kaggle scikit-learn seaborn

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_score, recall_score, f1_score, accuracy_score
)

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))


## 3. Dataset Download

### Option A — Kaggle API (recommended, works in Colab)

1. Go to [kaggle.com](https://www.kaggle.com) → Account → **Create New API Token**. This downloads a `kaggle.json` file.
2. Upload that file when prompted below.
3. Run the cell — it downloads and unzips the dataset automatically.

### Option B — Manual download

Download the dataset zip directly from Kaggle, upload it to your Colab/Jupyter environment as `casting_data.zip`, place it in the working directory, and skip straight to the "unzip" cell.


In [ ]:
# --- Option A: Kaggle API download (skip if you already have the data) ---
from google.colab import files

print("Please upload your kaggle.json API key file")
uploaded = files.upload()  # upload kaggle.json here

os.makedirs("/root/.kaggle", exist_ok=True)
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d ravirajsinh45/real-life-industrial-dataset-of-casting-product


In [ ]:
# Unzip the dataset (run this whether you used Option A or Option B)
import zipfile

zip_path = "real-life-industrial-dataset-of-casting-product.zip"
if not os.path.exists(zip_path):
    zip_path = "casting_data.zip"  # fallback name if manually uploaded

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("casting_data")

print("Extraction complete.")

# The dataset ships with a pre-made train/test split. We locate those folders below.
for root, dirs, files_ in os.walk("casting_data"):
    depth = root.count(os.sep)
    if depth <= 4:
        print(root)


In [ ]:
# Set these paths based on the folder structure printed above.
# The Kaggle dataset typically extracts to:
# casting_data/casting_data/train/{ok_front, def_front}
# casting_data/casting_data/test/{ok_front, def_front}

BASE_DIR = "casting_data/casting_data"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR = os.path.join(BASE_DIR, "test")

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
CLASS_NAMES = ["def_front", "ok_front"]  # 0 = defective, 1 = ok

print("Train dir exists:", os.path.exists(TRAIN_DIR))
print("Test dir exists:", os.path.exists(TEST_DIR))


## 4. Exploratory Data Analysis (EDA)

Before building any model, we look at:
- How many images are in each class (class balance)
- What the images actually look like
- Image dimensions/quality


In [ ]:
# Count images per class in train and test sets
def count_images(directory):
    counts = {}
    for class_name in os.listdir(directory):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            counts[class_name] = len(os.listdir(class_path))
    return counts

train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)

print("Training set:", train_counts)
print("Test set:", test_counts)

summary_df = pd.DataFrame({"train": train_counts, "test": test_counts}).fillna(0).astype(int)
summary_df["total"] = summary_df["train"] + summary_df["test"]
summary_df


In [ ]:
# Visualize class balance
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

summary_df[["train", "test"]].plot(kind="bar", ax=ax[0])
ax[0].set_title("Image Count per Class (Train vs Test)")
ax[0].set_ylabel("Number of images")
ax[0].tick_params(axis="x", rotation=0)

summary_df["total"].plot(kind="pie", autopct="%1.1f%%", ax=ax[1], colors=["#e74c3c", "#2ecc71"])
ax[1].set_title("Overall Class Distribution")
ax[1].set_ylabel("")

plt.tight_layout()
plt.show()


In [ ]:
# Visualize sample images from each class
def show_samples(directory, class_name, n=5):
    class_path = os.path.join(directory, class_name)
    sample_files = os.listdir(class_path)[:n]

    fig, axes = plt.subplots(1, n, figsize=(15, 3))
    for i, fname in enumerate(sample_files):
        img = tf.keras.utils.load_img(os.path.join(class_path, fname))
        axes[i].imshow(img)
        axes[i].axis("off")
    fig.suptitle(f"Sample images: {class_name}")
    plt.show()

show_samples(TRAIN_DIR, "ok_front")
show_samples(TRAIN_DIR, "def_front")


In [ ]:
# Check image dimensions to confirm consistency
sample_img_path = os.path.join(TRAIN_DIR, "ok_front", os.listdir(os.path.join(TRAIN_DIR, "ok_front"))[0])
sample_img = tf.keras.utils.load_img(sample_img_path)
print("Sample image size:", sample_img.size, "| mode:", sample_img.mode)


**EDA takeaways:**
- The dataset is close to balanced between `ok_front` and `def_front`, which means we generally do not need heavy class-imbalance handling, but we should still check the exact ratio (printed above) and keep an eye on recall for the defective class — in a QC setting, **missing a real defect (false negative) is usually more costly than a false alarm (false positive)**.
- Images are grayscale casting photos taken from a consistent camera angle, which makes this a good candidate for both a simple CNN and transfer learning.


## 5. Data Preprocessing & Augmentation

We use `ImageDataGenerator` to:
- Rescale pixel values to [0, 1]
- Apply light augmentation (rotation, zoom, flips) on the training set only, to help the model generalize and reduce overfitting on a relatively small dataset
- Hold out part of the training folder as a validation set


In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2  # 80% train / 20% validation
)

test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    color_mode="rgb",
    classes=CLASS_NAMES,
    subset="training",
    seed=SEED
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    color_mode="rgb",
    classes=CLASS_NAMES,
    subset="validation",
    seed=SEED
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    color_mode="rgb",
    classes=CLASS_NAMES,
    shuffle=False  # keep order for evaluation later
)

print("Class indices:", train_generator.class_indices)


## 6. Model 1 — Baseline CNN (Built From Scratch)

We start with a simple convolutional neural network as a baseline. This gives us something to compare the transfer-learning model against, and helps illustrate how much transfer learning actually helps on a small dataset.


In [ ]:
def build_baseline_cnn(input_shape=(128, 128, 3)):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(1, activation="sigmoid")  # binary output
    ])
    return model

baseline_model = build_baseline_cnn()
baseline_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)
baseline_model.summary()


In [ ]:
early_stop = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)

history_baseline = baseline_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25,
    callbacks=[early_stop, reduce_lr]
)


## 7. Model 2 — Transfer Learning (MobileNetV2)

Transfer learning reuses a model already trained on millions of images (ImageNet) and fine-tunes it on our smaller casting-defect dataset. This usually gives **better accuracy with less training data and less training time** than a CNN trained from scratch — a good fit here since our dataset has only a few thousand images.

We freeze the pretrained base first (feature extraction), then optionally unfreeze the top layers for fine-tuning.


In [ ]:
def build_transfer_model(input_shape=(128, 128, 3)):
    base_model = MobileNetV2(input_shape=input_shape, include_top=False, weights="imagenet")
    base_model.trainable = False  # freeze pretrained layers initially

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid")
    ])
    return model, base_model

transfer_model, base_model = build_transfer_model()
transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)
transfer_model.summary()


In [ ]:
# Phase 1: train the new top layers only (base model frozen)
history_transfer = transfer_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    callbacks=[early_stop, reduce_lr]
)


In [ ]:
# Phase 2: fine-tune — unfreeze the top layers of MobileNetV2 and train with a lower learning rate
base_model.trainable = True

# Freeze all but the last 30 layers so fine-tuning stays stable and fast
for layer in base_model.layers[:-30]:
    layer.trainable = False

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # much smaller LR for fine-tuning
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

history_finetune = transfer_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[early_stop, reduce_lr]
)


## 8. Training Curves

Compare how both models learned over time.

In [ ]:
def plot_history(history, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(history.history["accuracy"], label="train")
    ax[0].plot(history.history["val_accuracy"], label="val")
    ax[0].set_title(f"{title} — Accuracy")
    ax[0].set_xlabel("Epoch")
    ax[0].legend()

    ax[1].plot(history.history["loss"], label="train")
    ax[1].plot(history.history["val_loss"], label="val")
    ax[1].set_title(f"{title} — Loss")
    ax[1].set_xlabel("Epoch")
    ax[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(history_baseline, "Baseline CNN")
plot_history(history_transfer, "Transfer Learning (feature extraction)")
plot_history(history_finetune, "Transfer Learning (fine-tuning)")


## 9. Model Evaluation on the Test Set

We evaluate both models on the held-out **test set** (images the models never saw during training or validation) using metrics that matter for a QC use case:

- **Accuracy** — overall correctness
- **Precision** — of predicted defects, how many are truly defective
- **Recall** — of actual defects, how many did we catch (critical: missed defects are costly)
- **F1-score** — balance of precision and recall
- **ROC-AUC** — overall ability to separate the two classes
- **Confusion matrix** — a full breakdown of correct/incorrect predictions per class


In [ ]:
def evaluate_model(model, generator, model_name):
    generator.reset()
    y_true = generator.classes
    y_prob = model.predict(generator, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)

    print(f"=== {model_name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")
    print()
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    return {
        "model": model_name, "y_true": y_true, "y_pred": y_pred, "y_prob": y_prob,
        "accuracy": acc, "precision": prec, "recall": rec, "f1": f1,
        "roc_auc": roc_auc, "fpr": fpr, "tpr": tpr
    }

baseline_results = evaluate_model(baseline_model, test_generator, "Baseline CNN")
transfer_results = evaluate_model(transfer_model, test_generator, "Transfer Learning (MobileNetV2)")


In [ ]:
# Side-by-side comparison table
comparison_df = pd.DataFrame([
    {"Model": "Baseline CNN", "Accuracy": baseline_results["accuracy"], "Precision": baseline_results["precision"],
     "Recall": baseline_results["recall"], "F1-score": baseline_results["f1"], "ROC-AUC": baseline_results["roc_auc"]},
    {"Model": "Transfer Learning (MobileNetV2)", "Accuracy": transfer_results["accuracy"], "Precision": transfer_results["precision"],
     "Recall": transfer_results["recall"], "F1-score": transfer_results["f1"], "ROC-AUC": transfer_results["roc_auc"]},
]).set_index("Model").round(4)

comparison_df


In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, res, title in zip(axes, [baseline_results, transfer_results], ["Baseline CNN", "Transfer Learning"]):
    cm = confusion_matrix(res["y_true"], res["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES,
                yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f"Confusion Matrix — {title}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


In [ ]:
# ROC curves comparison
plt.figure(figsize=(6, 6))
plt.plot(baseline_results["fpr"], baseline_results["tpr"],
         label=f"Baseline CNN (AUC = {baseline_results[\'roc_auc\']:.3f})")
plt.plot(transfer_results["fpr"], transfer_results["tpr"],
         label=f"Transfer Learning (AUC = {transfer_results[\'roc_auc\']:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()


## 10. Error Analysis — Misclassified Images

Looking at what the best model gets wrong helps explain its limitations and where a human inspector should still double-check the model's decision.


In [ ]:
# Pick the better-performing model based on F1-score for error analysis
best_results = transfer_results if transfer_results["f1"] >= baseline_results["f1"] else baseline_results
best_model_name = best_results["model"]
print("Best model for error analysis:", best_model_name)

test_generator.reset()
filepaths = test_generator.filepaths
y_true = best_results["y_true"]
y_pred = best_results["y_pred"]

misclassified_idx = np.where(y_true != y_pred)[0]
print(f"Number of misclassified test images: {len(misclassified_idx)} / {len(y_true)}")

n_show = min(6, len(misclassified_idx))
if n_show > 0:
    fig, axes = plt.subplots(1, n_show, figsize=(15, 3))
    if n_show == 1:
        axes = [axes]
    for i, idx in enumerate(misclassified_idx[:n_show]):
        img = tf.keras.utils.load_img(filepaths[idx])
        axes[i].imshow(img)
        axes[i].axis("off")
        axes[i].set_title(f"True: {CLASS_NAMES[y_true[idx]]}\nPred: {CLASS_NAMES[y_pred[idx]]}", fontsize=9)
    plt.suptitle(f"Misclassified Examples — {best_model_name}")
    plt.tight_layout()
    plt.show()
else:
    print("No misclassified images to display.")


## 11. Saving & Exporting the Final Model

We save the better-performing model in two formats:
- **Keras format (`.keras`)** for reloading in Python/TensorFlow later
- A small **inference helper function** that takes a raw image path and returns a prediction, ready to plug into an application


In [ ]:
best_model = transfer_model if best_model_name.startswith("Transfer") else baseline_model

MODEL_PATH = "casting_defect_classifier.keras"
best_model.save(MODEL_PATH)
print(f"Saved best model ({best_model_name}) to {MODEL_PATH}")


In [ ]:
def predict_image(model, image_path, img_size=IMG_SIZE, class_names=CLASS_NAMES, threshold=0.5):
    """Load an image, preprocess it, and return the predicted class + confidence."""
    img = tf.keras.utils.load_img(image_path, target_size=img_size)
    img_array = tf.keras.utils.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prob = model.predict(img_array, verbose=0)[0][0]
    pred_class = class_names[int(prob >= threshold)]
    confidence = prob if prob >= threshold else 1 - prob

    return pred_class, float(confidence)

# Example usage (swap in any test image path):
example_path = test_generator.filepaths[0]
pred_class, confidence = predict_image(best_model, example_path)
print(f"Image: {example_path}")
print(f"Prediction: {pred_class} (confidence: {confidence:.2%})")


In [ ]:
# Reload check — confirms the saved model works correctly after loading fresh
reloaded_model = tf.keras.models.load_model(MODEL_PATH)
pred_class, confidence = predict_image(reloaded_model, example_path)
print(f"Reloaded model prediction: {pred_class} (confidence: {confidence:.2%})")


## 12. Business Summary & Recommendations

**What we built:** An automated binary image classifier that flags manufactured casting products as `ok_front` (acceptable) or `def_front` (defective) directly from a photo, without requiring a human inspector to make every call.

**Key findings:**
- The transfer-learning model (MobileNetV2) generally outperforms the from-scratch CNN baseline on this dataset size, converges faster, and needs fewer epochs to reach strong accuracy — see the comparison table and ROC curves above for exact numbers on this run.
- Recall on the defective class is the metric to watch most closely in production: a missed defect (false negative) reaching a customer is typically far more costly than a false alarm that sends a good product for a second look.

**Recommended next steps for deployment:**
1. **Set the decision threshold deliberately.** The default 0.5 cutoff balances precision and recall equally — in most QC settings you'd lower the threshold to catch more true defects, accepting a few more false alarms in exchange.
2. **Route borderline predictions to a human.** For images where the model's confidence is close to 50%, send them for manual review rather than auto-approving or auto-rejecting.
3. **Keep monitoring after deployment.** Retrain periodically as new defect types or camera/lighting changes appear on the line — this is called *model drift* and is common in manufacturing.
4. **Pilot before full rollout.** Run the model alongside human inspectors for a trial period, comparing decisions before removing manual review entirely.

**Limitations of this notebook:**
- Trained on a single product type (submersible pump impellers) — performance on other products would need a separate evaluation.
- The dataset reflects a fixed camera angle and lighting; real production lines with more variation would likely need more diverse training images and additional augmentation.
